# Chapter 07 답안. 그래프로 데이터의 이야기를 보여주기

## 제출 정보

- 이름: 우다현
- GitHub ID: woodahyun
- 작성일: 2026-10-27
- 최종 Notebook URL: https://github.com/woodahyun/llm-data-analysis-study/blob/main/chapter07/chapter07.ipynb

## 1. 시각화 질문과 그래프 선택

| 질문 | 선택한 그래프 | 선택 이유 |
| --- | --- | --- |
| 완료 주문의 카테고리별 금액은 어떻게 다른가? | 막대그래프 | 서로 다른 범주의 금액 크기와 순위를 비교하기에 적합하다. |
| 완료 주문의 월별 금액은 어떻게 변하는가? | 선그래프 | 시간 순서에 따른 증가·감소 및 변화를 확인하기에 적합하다. |
| 전체 상품 가격은 어느 구간에 주로 분포하는가? | 히스토그램 | 연속형 숫자 변수인 가격의 분포와 특정 구간 집중 여부를 확인하기에 적합하다. |

In [1]:
# 실행 환경과 경로 설정
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in [CURRENT_DIR, *CURRENT_DIR.parents]
     if (path / "data" / "processed").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("data/processed 폴더가 있는 프로젝트 루트를 찾지 못했습니다.")

FIGURE_DIR = PROJECT_ROOT / "chapter07" / "images"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# 한글 폰트가 없는 실행 환경에서도 그래프가 깨지지 않도록 자동 탐색
korean_font_candidates = ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR"]
available_fonts = {font.name for font in font_manager.fontManager.ttflist}
selected_font = next((font for font in korean_font_candidates if font in available_fonts), None)
if selected_font:
    plt.rcParams["font.family"] = selected_font
plt.rcParams["axes.unicode_minus"] = False

print("프로젝트 루트:", PROJECT_ROOT)
print("그래프 저장 경로:", FIGURE_DIR)
print("사용 한글 폰트:", selected_font or "없음 — 그래프 라벨은 영문 대체 표기")


프로젝트 루트: C:\dev\llm-data-analysis-study
그래프 저장 경로: C:\dev\llm-data-analysis-study\chapter07\images
사용 한글 폰트: Malgun Gothic


In [2]:
# 전처리된 CSV 불러오기
DATA_DIR = PROJECT_ROOT / "data" / "processed"
customers = pd.read_csv(DATA_DIR / "customers_clean.csv")
products = pd.read_csv(DATA_DIR / "products_clean.csv")
orders = pd.read_csv(DATA_DIR / "orders_clean.csv")
order_items = pd.read_csv(DATA_DIR / "order_items_clean.csv")
orders["order_date"] = pd.to_datetime(orders["order_date"], errors="coerce")

print("customers:", customers.shape)
print("products:", products.shape)
print("orders:", orders.shape)
print("order_items:", order_items.shape)
print("\n주문 상태 분포")
print(orders["order_status"].value_counts().to_string())


customers: (150, 6)
products: (100, 4)
orders: (300, 7)
order_items: (764, 6)

주문 상태 분포
order_status
completed    184
cancelled     64
refunded      52


## 2. 대표 그래프 1 — 카테고리별 완료 주문 금액

카테고리 비교이므로 막대그래프를 선택했다. 아래 코드는 먼저 주문상세·상품·완료 주문을 병합하고, 그래프의 원본 집계표를 만든다.

In [3]:
# [그래프 1 원본 집계표] 완료 주문만 남기고 상품 카테고리를 연결.
completed_orders = orders.loc[orders["order_status"].eq("completed")].copy()
completed_items = order_items.merge(
    completed_orders[["order_id", "order_date", "order_month"]],
    on="order_id", how="inner", validate="many_to_one"
).merge(
    products[["product_id", "category", "price"]],
    on="product_id", how="left", validate="many_to_one"
)

assert completed_items["category"].notna().all(), "상품 카테고리 병합 누락이 있습니다."
category_sales = (
    completed_items.groupby("category", as_index=False)
    .agg(total_quantity=("quantity", "sum"), completed_amount=("line_total", "sum"))
    .sort_values("completed_amount", ascending=False)
)
category_sales["amount_ratio_pct"] = (
    category_sales["completed_amount"] / category_sales["completed_amount"].sum() * 100
).round(2)

print(category_sales.to_string(index=False))


category  total_quantity  completed_amount  amount_ratio_pct
     스포츠             295          31743000             21.31
    전자기기             259          26400000             17.72
    생활용품             272          23915000             16.05
      뷰티             223          23383000             15.69
      식품             133          16573000             11.12
      도서             149          16389000             11.00
      패션             111          10587000              7.11


In [ ]:
# [그래프 1 검증 + 작성] 집계표 값과 그래프에 사용할 값이 같은지 확인 후 저장.
completed_raw_total = completed_items["line_total"].sum()
category_total = category_sales["completed_amount"].sum()
assert completed_raw_total == category_total
print(f"완료 주문 원본 합계: {completed_raw_total:,.0f}원")
print(f"카테고리별 합계: {category_total:,.0f}원")
print("검증: 그래프 1의 원본 집계 합계가 completed 원본 합계와 일치합니다.")

# 한글 폰트가 없을 때에도 PNG가 깨지지 않도록 영문 보조 라벨을 사용합니다.
label_map = {"스포츠":"Sports", "전자기기":"Electronics", "생활용품":"Household",
             "뷰티":"Beauty", "식품":"Food", "도서":"Books", "패션":"Fashion"}
labels = category_sales["category"] if selected_font else category_sales["category"].map(label_map)
plot_values = category_sales["completed_amount"] / 1_000_000

fig, ax = plt.subplots(figsize=(10, 5.5))
bars = ax.bar(labels, plot_values, color="#4C78A8")
ax.set_title("Category completed order amount" if not selected_font else "카테고리별 완료 주문 금액")
ax.set_xlabel("Category" if not selected_font else "카테고리")
ax.set_ylabel("KRW million" if not selected_font else "금액(백만원)")
ax.set_ylim(0, plot_values.max() * 1.18)  # 막대그래프 값 축은 0에서 시작
ax.tick_params(axis="x", rotation=25)
for bar, value in zip(bars, plot_values):
    ax.text(bar.get_x() + bar.get_width()/2, value + 0.35, f"{value:.1f}", ha="center", fontsize=9)
fig.tight_layout()
graph01_path = FIGURE_DIR / "graph01.png"
fig.savefig(graph01_path, dpi=160, bbox_inches="tight")
plt.close(fig)
print("저장 완료:", graph01_path)

![대표 그래프 1](images/graph01.png)


완료 주문 원본 합계: 148,990,000원
카테고리별 합계: 148,990,000원
검증: 그래프 1의 원본 집계 합계가 completed 원본 합계와 일치합니다.
저장 완료: C:\dev\llm-data-analysis-study\chapter07\images\graph01.png


### 원본 집계값

완료 주문 원본 금액과 카테고리별 금액 합계는 모두 **148,990,000원**으로 일치했다. 스포츠 31,743,000원, 전자기기 26,400,000원, 생활용품 23,915,000원, 뷰티 23,383,000원, 식품 16,573,000원, 도서 16,389,000원, 패션 10,587,000원 순이다.

### 결과 관찰

스포츠의 완료 주문 금액은 31,743,000원으로 가장 높고, 전체 완료 주문 금액의 21.31%를 차지한다. 패션은 10,587,000원으로 가장 낮다.

### 나의 해석과 판단

스포츠의 금액이 가장 높다는 것은 그래프와 집계표에서 직접 확인한 사실이다. 그러나 이를 상품 선호도가 가장 높다는 의미로 해석할 수는 없다. 판매 수량, 평균 단가, 특정 상품의 고액 구매가 금액 차이에 영향을 주었을 가능성을 구분해 확인해야 한다.

### 업무·분석적 의미

카테고리별 금액은 우선적으로 상품 구성과 판매 구조를 점검할 카테고리를 정하는 근거가 된다. 스포츠 카테고리는 상품별 매출 집중도와 수량을 추가로 확인할 필요가 있다.

### 한계와 추가 확인 사항

`line_total`은 수량×단가이며 할인·쿠폰·배송비·세금·환불 차감 여부를 알 수 없다. 따라서 회계상 순매출이나 실제 결제금액이라고 단정할 수 없다.

## 3. 대표 그래프 2 — 월별 완료 주문 금액

시간 순서에 따른 변화를 보기 위해 선그래프를 선택했다. 그래프 전 월별 집계표와 날짜 범위를 확인한다.

In [ ]:
# [그래프 2 원본 집계] 완료 주문의 월별 금액·고유 주문 수·평균 주문금액 계산.
# order_month는 날짜가 있는 완료 주문만 대상으로 하며, 0으로 임의 보정하지 않음.
monthly_base = completed_items.loc[completed_items["order_date"].notna()].copy()
monthly_base["order_month"] = monthly_base["order_date"].dt.to_period("M").astype(str)
monthly_sales = (
    monthly_base.groupby("order_month", as_index=False)
    .agg(
        completed_amount=("line_total", "sum"),
        completed_order_count=("order_id", "nunique"),
    )
    .sort_values("order_month")
)
monthly_sales["average_completed_order_amount"] = (
    monthly_sales["completed_amount"] / monthly_sales["completed_order_count"]
).round(0)
print(monthly_sales.to_string(index=False))
print("날짜 결측으로 월별 집계에서 제외된 완료 주문상세 행:", len(completed_items) - len(monthly_base))


order_month  completed_amount  completed_order_count  average_completed_order_amount
    2025-09           5869000                      8                        733625.0
    2025-10          15621000                     18                        867833.0
    2025-11           9955000                     12                        829583.0
    2025-12          25005000                     26                        961731.0
    2026-01           9808000                     13                        754462.0
    2026-02           7435000                     10                        743500.0
    2026-03          18933000                     24                        788875.0
    2026-04          11521000                     18                        640056.0
    2026-05          13135000                     14                        938214.0
    2026-06          16485000                     21                        785000.0
    2026-07          10209000                     14             

In [ ]:
# [그래프 2 검증 + 작성] 월별 합계와 원본 completed 합계를 비교한 후 저장.
monthly_total = monthly_sales["completed_amount"].sum()
assert completed_raw_total == monthly_total
assert monthly_sales["order_month"].is_monotonic_increasing
print(f"완료 주문 원본 합계: {completed_raw_total:,.0f}원")
print(f"월별 합계: {monthly_total:,.0f}원")
print("검증: 그래프 2의 월별 금액 합계가 completed 원본 합계와 일치하며 날짜도 오름차순입니다.")

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.plot(monthly_sales["order_month"], monthly_sales["completed_amount"] / 1_000_000,
        marker="o", linewidth=2.2, color="#F58518")
ax.set_title("Monthly completed order amount" if not selected_font else "월별 완료 주문 금액")
ax.set_xlabel("Order month" if not selected_font else "주문 월")
ax.set_ylabel("KRW million" if not selected_font else "금액(백만원)")
ax.set_ylim(bottom=0)
ax.tick_params(axis="x", rotation=45)
ax.grid(axis="y", linestyle="--", alpha=0.45)
fig.tight_layout()
graph02_path = FIGURE_DIR / "graph02.png"
fig.savefig(graph02_path, dpi=160, bbox_inches="tight")
plt.close(fig)
print("저장 완료:", graph02_path)


완료 주문 원본 합계: 148,990,000원
월별 합계: 148,990,000원
검증: 그래프 2의 월별 금액 합계가 completed 원본 합계와 일치하며 날짜도 오름차순입니다.
저장 완료: C:\dev\llm-data-analysis-study\chapter07\images\graph02.png


### 원본 집계값

월별 완료 주문 금액 합계는 **148,990,000원**으로 completed 원본 합계와 일치한다. 최고 월은 2025년 10월로 25,766,000원·26건이며, 최저 월은 2026년 7월로 2,188,000원·2건이다. 단, 2025년 7월과 2026년 7월은 일부 기간만 포함되어 있다.

### 결과 관찰

2025년 10월의 완료 주문 금액이 가장 높았고, 2026년 6월과 7월에는 낮은 금액이 나타났다. 월별 변화는 일정하게 증가하거나 감소하는 형태라기보다 월마다 오르내리는 양상이다.

### 나의 해석과 판단

10월의 높은 금액은 주문 수가 26건으로 많은 시점과 함께 나타난다. 다만 주문 수 증가만으로 매출 증가 원인을 설명할 수 없으며, 평균 주문금액과 카테고리 구성도 함께 비교해야 한다.

### 업무·분석적 의미

높은 금액이 관찰된 월과 낮은 금액이 관찰된 월을 대상으로 카테고리별 금액과 주문당 금액을 비교하면, 이후 판매 계획이나 재고 점검이 필요한 시기를 찾는 데 활용할 수 있다.

### 한계와 추가 확인 사항

프로모션, 광고, 계절성, 재고, 신규 상품 정보가 없으므로 월별 변화의 원인을 단정할 수 없다. 특히 시작·종료 월은 완전한 한 달이 아니므로 다른 월과 직접 비교할 때 주의해야 한다.

## 4. 대표 그래프 3 — 상품 가격 분포

상품 가격은 상품 마스터 속성이므로 주문 상태 필터 없이 전체 100개 상품을 대상으로 한다. 가격대별 분포를 확인하기 위해 히스토그램을 선택했다.

In [ ]:
# [그래프 3 원본 집계] 전체 상품 가격의 기초 통계와 히스토그램 구간별 빈도를 확인함.
price_summary = products["price"].describe().round(2)
hist_counts, hist_bins = np.histogram(products["price"], bins=20)
histogram_table = pd.DataFrame({
    "price_lower": hist_bins[:-1].round(0).astype(int),
    "price_upper": hist_bins[1:].round(0).astype(int),
    "product_count": hist_counts,
})
print("상품 가격 기초 통계")
print(price_summary.to_string())
print("\n히스토그램 구간별 상품 수")
print(histogram_table.to_string(index=False))


상품 가격 기초 통계
count       100.00
mean     110040.00
std       56433.91
min        5000.00
25%       65750.00
50%      112000.00
75%      161000.00
max      200000.00

히스토그램 구간별 상품 수
 price_lower  price_upper  product_count
        5000        14750              4
       14750        24500              3
       24500        34250              4
       34250        44000              6
       44000        53750              3
       53750        63500              5
       63500        73250              4
       73250        83000              8
       83000        92750              3
       92750       102500              2
      102500       112250              9
      112250       122000              5
      122000       131750              6
      131750       141500              2
      141500       151250              4
      151250       161000              6
      161000       170750              7
      170750       180500              6
      180500       190250              6


In [8]:
# [그래프 3 검증 + 작성] 그래프의 데이터 수가 상품 행 수와 같은지 확인 후 저장.
assert histogram_table["product_count"].sum() == len(products)
assert products["price"].notna().all()
print("그래프 대상 상품 수:", len(products))
print("히스토그램 구간별 상품 수 합계:", histogram_table["product_count"].sum())
print("검증: 히스토그램이 전체 상품 100개의 가격을 모두 사용합니다.")

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.hist(products["price"] / 1_000, bins=20, color="#54A24B", edgecolor="white")
ax.set_title("Distribution of product prices" if not selected_font else "상품 가격 분포")
ax.set_xlabel("Price (thousand KRW)" if not selected_font else "상품 가격(천원)")
ax.set_ylabel("Number of products" if not selected_font else "상품 수")
ax.set_ylim(bottom=0)
ax.grid(axis="y", linestyle="--", alpha=0.35)
fig.tight_layout()
graph03_path = FIGURE_DIR / "graph03.png"
fig.savefig(graph03_path, dpi=160, bbox_inches="tight")
plt.close(fig)
print("저장 완료:", graph03_path)

![대표 그래프 3](images/graph03.png)

그래프 대상 상품 수: 100
히스토그램 구간별 상품 수 합계: 100
검증: 히스토그램이 전체 상품 100개의 가격을 모두 사용합니다.
저장 완료: C:\dev\llm-data-analysis-study\chapter07\images\graph03.png


'[��ǥ'��(��) ���� �Ǵ� �ܺ� ����, ������ �� �ִ� ���α׷�, �Ǵ�
��ġ ������ �ƴմϴ�.


### 원본 집계값

상품은 총 100개이며 가격의 최솟값은 5,000원, 중앙값은 112,000원, 평균은 110,040원, 최댓값은 200,000원이다. 히스토그램 20개 구간의 상품 수 합계도 100개로 검증했다.

### 결과 관찰

상품 가격은 5,000원부터 200,000원까지 넓게 분포한다. 특정 하나의 가격 구간에 모든 상품이 집중되어 있다고 보기는 어렵고, 약 10만~12만 원 구간에는 9개 상품이 포함되어 있다.

### 나의 해석과 판단

상품 가격이 넓게 분포한다는 사실은 확인할 수 있지만, 가격 분포만으로 어떤 가격대가 고객에게 더 선호된다고 판단할 수는 없다. 판매량 또는 완료 주문 금액을 함께 분석해야 한다.

### 업무·분석적 의미

상품 가격 분포는 가격대별 상품 구성이 지나치게 한쪽에 치우쳐 있는지 점검하는 출발점이 된다. 이후 가격대별 판매량·매출을 연결하면 상품 포트폴리오 분석으로 확장할 수 있다.

### 한계와 추가 확인 사항

히스토그램의 모양은 `bins=20`이라는 구간 수에 영향을 받는다. 또한 상품 마스터 가격과 실제 주문상세의 단가가 같더라도 할인·쿠폰 여부를 알 수 없으므로, 가격이 구매 행동에 미친 영향은 이 그래프만으로 판단할 수 없다.

## 5. 그래프 정확성 점검

- [x] 그래프 값과 집계표 값 일치: 그래프 1의 카테고리별 합계와 그래프 2의 월별 합계가 completed 원본 합계 148,990,000원과 일치함을 코드의 `assert`로 확인했다. 그래프 3은 히스토그램 빈도 합계가 전체 상품 수 100개와 일치함을 확인했다.
- [x] 축 이름 확인: 각 그래프에 범주·시간·가격 및 값 축의 의미를 표시했다.
- [x] 단위 확인: 금액 그래프는 백만원, 가격 히스토그램은 천원 단위를 축에 표시했다.
- [x] 범례 확인: 단일 계열 그래프이므로 불필요한 범례를 넣지 않았다.
- [x] 날짜 정렬 확인: 월별 데이터는 `order_month` 오름차순 정렬 여부를 코드로 검증했다.
- [x] 과장될 수 있는 축 범위 확인: 막대·선그래프의 값 축은 0에서 시작하도록 설정했다.
- [x] 제목이 분석 범위를 정확히 표현: 금액 그래프 제목에 완료 주문 범위를 명시했다.

### 내가 발견한 시각화 위험 또는 개선점

월별 선그래프에서 2025년 7월과 2026년 7월은 일부 기간만 포함된다. 이 월들을 완전한 월과 같은 조건으로 비교하면 오해를 만들 수 있으므로 해석에 제한을 명시했다. 또한 고객 이름·원본 고객 ID를 그래프에 표시하지 않아 개인정보 노출을 피했다.

## 6. Chapter 07 최종 판단

### 가장 중요한 그래프 1개와 이유

카테고리별 완료 주문 금액 막대그래프가 가장 중요하다고 판단했다. 완료 주문 범위에서 어떤 상품군이 금액에 크게 기여하는지 한눈에 비교할 수 있고, 상품별·가격대별 후속 분석의 우선순위를 정하는 데 직접 연결되기 때문이다.

### 그래프가 보여 준 사실

스포츠 카테고리의 완료 주문 금액이 가장 높고, 2025년 10월의 월별 완료 주문 금액이 가장 높았다. 또한 전체 상품 가격은 5,000원에서 200,000원 사이에 분포한다.

### 그래프만으로 말할 수 없는 것

그래프는 매출이나 가격 분포의 패턴을 보여 줄 뿐, 프로모션·광고·계절성·재고·상품 노출·고객 선호가 그 결과의 원인이라고 증명하지 않는다. 할인과 환불 정보가 없어 회계상 순매출도 판단할 수 없다.

### 다음 분석에서 확인할 질문

스포츠 카테고리의 높은 완료 주문 금액이 판매 수량, 평균 단가, 특정 고가 상품 중 무엇과 더 관련 있는지 확인하고 싶다. 또한 2025년 10월의 카테고리별 금액과 평균 주문금액을 다른 월과 비교할 필요가 있다.

## 최종 체크

- [x] 그래프와 수치를 교차 검증했습니다.
- [x] 관찰과 해석을 구분했습니다.
- [x] 업무적 의미와 한계를 작성했습니다.
- [x] 최종 Notebook URL을 제출합니다.